# Lesson 5: Citation Networks - Setup

**Duration:** ~5 minutes  
**Module:** 5 - GDS with Python  
**Dataset:** Cora Citation Network

## What You'll Learn

- What citation networks reveal about scientific research
- How to load the Cora citation dataset
- How to create a graph projection for analysis

## What is a Citation Network?

Citation networks reveal:
- **Influential papers:** Which research shaped entire fields?
- **Bridge papers:** Which works connect different research areas?
- **Research communities:** Which papers form natural clusters?

**The Cora Dataset:**
- 2,708 academic papers
- 7 subject areas (Neural Networks, Reinforcement Learning, Theory, Genetic Algorithms, Case-Based Reasoning, Probabilistic Methods, Rule Learning)
- 10,556 citation relationships (Paper A cites Paper B)
- Each paper has content features (1,433-dimensional vector representing words used)

**Graph structure:**
- Nodes: `Paper` (with properties: `subject`, `features`, `subjectClass`)
- Relationships: `CITES` (directed: Paper A -> Paper B means A cites B)

## Part 1: Connect to Neo4j

In [2]:
import os
import pandas as pd
from IPython.display import display
from graphdatascience import GraphDataScience
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get Neo4j credentials from environment variables
uri = os.getenv('NEO4J_URI')
username = os.getenv('NEO4J_USERNAME')
password = os.getenv('NEO4J_PASSWORD')

# Create GDS client
gds = GraphDataScience(uri, auth=(username, password))

# Verify connection and check GDS version
print(f"Connected to GDS version: {gds.version()}")
print(f"Connected to: {uri}")

Connected to GDS version: 2.25.0
Connected to: neo4j://127.0.0.1:7687


## Part 2: Load the Citation Network

Load the Cora citation dataset from GitHub.

In [3]:
# Load paper nodes (2,708 papers with subjects and features)
node_load_q = """
LOAD CSV WITH HEADERS FROM 
    "https://raw.githubusercontent.com/Kristof-Neys/Neo4j-Cora/main/node_list.csv" AS row
WITH toInteger(row.id) AS paperId, 
     row.subject AS subject, 
     row.features AS features
MERGE (p:Paper {paper_Id: paperId})
SET p.subject = subject, 
    p.features = apoc.convert.fromJsonList(features)
RETURN count(p) AS papers_loaded
"""

result = gds.run_cypher(node_load_q)
print(f"Loaded {result['papers_loaded'][0]} papers")

Loaded 2708 papers


In [4]:
# Load citation relationships (10,556 citations)
edge_load_q = """
LOAD CSV WITH HEADERS FROM 
    "https://raw.githubusercontent.com/Kristof-Neys/Neo4j-Cora/main/edge_list.csv" AS row
MATCH (source:Paper {paper_Id: toInteger(row.source)})
MATCH (target:Paper {paper_Id: toInteger(row.target)})
MERGE (source)-[r:CITES]->(target)
RETURN count(r) AS citations_loaded
"""

result = gds.run_cypher(edge_load_q)
print(f"Loaded {result['citations_loaded'][0]} citations")

Loaded 5429 citations


In [5]:
# Quick peek at the data
q_peek = """
MATCH (n:Paper) 
WHERE n.features IS NOT NULL 
RETURN n.paper_Id AS PaperId, 
       n.subject AS Paper_Subject, 
       size(n.features) AS num_features
LIMIT 5
"""

df = gds.run_cypher(q_peek)
print("Sample papers from dataset:")
display(df)

Sample papers from dataset:


,PaperId,Paper_Subject,num_features
0,31336,Neural_Networks,1433
1,1061127,Rule_Learning,1433
2,1106406,Reinforcement_Learning,1433
3,13195,Reinforcement_Learning,1433
4,37879,Probabilistic_Methods,1433


In [6]:
# Convert subject labels to numeric values (needed for some algorithms)
query_subj = """
MATCH (p:Paper)
WITH collect(DISTINCT p.subject) AS listSubjects
WITH listSubjects, size(listSubjects) AS sizeListSubjects
WITH listSubjects, range(1, sizeListSubjects) AS rangeSubjects
WITH apoc.map.fromLists(listSubjects, rangeSubjects) AS mapSubjects
MATCH (p:Paper)
SET p.subjectClass = mapSubjects[p.subject]
RETURN count(p) AS papers_updated
"""

result = gds.run_cypher(query_subj)
print(f"Converted subjects to numeric for {result['papers_updated'][0]} papers")

Converted subjects to numeric for 2708 papers


In [7]:
# Verify: check papers by subject
df = gds.run_cypher("""
    MATCH (p:Paper)
    RETURN p.subject AS subject, count(*) AS count
    ORDER BY count DESC
""")
print("Papers by subject:")
display(df)

Papers by subject:


,subject,count
0,Neural_Networks,818
1,Probabilistic_Methods,426
2,Genetic_Algorithms,418
3,Theory,351
4,Case_Based,298
5,Reinforcement_Learning,217
6,Rule_Learning,180


## Part 3: Create Graph Projection

Before running algorithms, we create an **in-memory graph projection**.

**Why project?**
- Algorithms run on optimized in-memory structure (much faster)
- Can include only relevant nodes/relationships
- Can configure orientation (directed vs undirected)
- Can include properties needed by algorithms

In [8]:
# Clean up any existing projections
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")

In [9]:
# Create a graph projection
G, result = gds.graph.project(
    "cora-graph",
    {
        "Paper": {
            "properties": ["subjectClass"]
        }
    },
    {
        "CITES": {
            "orientation": "UNDIRECTED"
        }
    }
)

# Inspect the projected graph
print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")
print(f"  Memory usage: {G.memory_usage()}")
print(f"  Density: {G.density():.6f}")
print(f"  Node properties: {G.node_properties('Paper')}")

Projected graph: cora-graph
  Nodes: 2,708
  Relationships: 10,858
  Memory usage: 932 KiB
  Density: 0.001481
  Node properties: ['subjectClass']


**Why UNDIRECTED?**

Even though citations are directional (Paper A cites Paper B), for influence analysis we treat them as connections in both directions. This captures the network of **related research** rather than just influence flow.

## Summary

You've set up the citation network for analysis:

- **Loaded** 2,708 papers and 10,556 citations
- **Converted** subject labels to numeric values
- **Created** an undirected graph projection

The projection `G` is now ready for algorithm analysis.

### Next Notebook

Run PageRank to find the most influential papers in the citation network.